# 02 - Friction Frame (v3, generated 2026-10-13)

Fabric folder: Dev_CGP > Notebooks > 03 - EADM > Collaboration. Run order: 01 - Policy Renewals 30d, then this notebook. v1 was "XX - Friction Extraction V3 - Frame"; v2 adds Home
change episodes and attaches each change call / event to one term per policy (v1 had 288 duplicate change ids);
v3 picks that term per policy and date whatever the event (v2 still had 70 duplicates).

The episode frame for the friction extract (spec v3.3), on the ±30-day renewal definition. One row per **First year**
or **Renewal** episode for Car, Home and Van, with every episode-level field and both outcome fields.
Reads `stg.seg_policy_renewals_30d` (run **SEG - Policy Renewals 30d** first). Writes `stg.fe3_*` only.

| Field | Rule |
|---|---|
| Renewal episode | one per policy term reaching its renewal: DueDate - 60 to DueDate + 14 |
| First-year episode | a genuinely new policy (a New Business sale that is not a renewal or rebook replacement): from the first quote linked to the sale (online: the bought MFQ quote's first event, or Home's HEQ quote; back office: the quote created with the Relay policy; otherwise the earliest acquisition event), no earlier than 30 days before the sale, to the first renewal's DueDate - 60 |
| Outcome_ThisRenewal | renewal episodes: Renewed, Lapsed, Cancelled, Not yet known (±30-day matching) |
| Outcome_NextRenewal | every episode: the policy's next renewal (the next term in its chain); Cancelled before renewal or Rebooked if the term was cancelled mid-term; blank after a Lapsed / Cancelled renewal |
| DecisionDate / Type | renewed: new policy's sale date, Accepted; cancelled: cancellation date, Cancelled; not renewed: DueDate, DatePassed; not yet known: EpisodeEnd, WindowEnd |
| PolicyChainRef | root policy code and root due date: the same across every renewal and rebook of the policy |
| DataDate | the earlier of the latest sale and the latest event outside the renewal steps (the stream holds synthetic renewal events dated into 2027) |

Episodes start from `FirstEpisodeDate` (24 months before the first snapshot, 202406). Snapshots are picked when
reading, by EpisodeStart.


Parameters. FirstEpisodeDate: 24 months before the first snapshot window (202406 = 2023-07-01 to 2024-06-30).

In [ ]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "ChillLH"
    }
}

In [1]:
Set FirstEpisodeDate = '2022-07-01' ;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 2, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 2 fields>

DataDate: the date the data runs to, the same on every row.

In [2]:
CREATE OR REPLACE TABLE stg.fe3_datadate AS
SELECT least((SELECT MAX(ReportingSaleDate) FROM edw.tbl_fact_policy_sales),
             (SELECT MAX(EventDate) FROM ods.episodeeventsnumberedfinal
              WHERE  EpisodeType <> 'renewals' AND EventDate <= current_date())) AS DataDate
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 3, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Policy terms of Car / Home / Van with the product named as in the spec, and a term index within each chain.

In [3]:
CREATE OR REPLACE TABLE stg.fe3_terms AS
SELECT r.*,
       CASE r.product WHEN 'Motor' THEN 'Car' ELSE r.product END              AS ProductName,
       concat(r.root_policy_code, '|', CAST(r.root_due_date AS STRING))       AS PolicyChainRef
FROM   stg.seg_policy_renewals_30d r
WHERE  r.product IN ('Motor', 'Home', 'Van')
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 4, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Each term's own renewal outcome, decision and the outcome of the next term in its chain.

In [4]:
CREATE OR REPLACE TABLE stg.fe3_term_outcome AS
WITH dd AS (SELECT DataDate FROM stg.fe3_datadate),
t AS (
    SELECT t.*,
           CASE WHEN t.due_flag = 0 AND t.rebooked = 1                       THEN 'Rebooked'
                WHEN t.due_flag = 0                                          THEN 'Cancelled before renewal'
                WHEN t.retained = 1                                          THEN 'Renewed'
                WHEN t.outcome_status = 'pending'                            THEN 'Not yet known'
                WHEN t.end_type = 'cancelled near renewal'                   THEN 'Cancelled'
                ELSE 'Lapsed' END                                                    AS term_outcome,
           CASE WHEN t.due_flag = 0                                          THEN t.cancel_date
                WHEN t.retained = 1                                          THEN t.new_term_start
                WHEN t.outcome_status = 'pending'                            THEN CAST(NULL AS DATE)
                WHEN t.end_type = 'cancelled near renewal'                   THEN t.cancel_date
                ELSE t.due_date END                                                  AS term_outcome_date
    FROM   stg.fe3_terms t
)
SELECT t.*,
       CASE WHEN t.due_flag = 0 THEN CAST(NULL AS DATE)
            WHEN t.retained = 1 THEN t.new_sale_date
            WHEN t.outcome_status = 'pending' THEN date_add(t.due_date, 14)
            WHEN t.end_type = 'cancelled near renewal' THEN t.cancel_date
            ELSE t.due_date END                                                      AS DecisionDate,
       CASE WHEN t.due_flag = 0 THEN CAST(NULL AS STRING)
            WHEN t.retained = 1 THEN 'Accepted'
            WHEN t.outcome_status = 'pending' THEN 'WindowEnd'
            WHEN t.end_type = 'cancelled near renewal' THEN 'Cancelled'
            ELSE 'DatePassed' END                                                    AS DecisionType,
       n.policy_code AS next_policy_code, n.due_date AS next_due_date,
       n.term_outcome AS next_term_outcome, n.term_outcome_date AS next_term_outcome_date
FROM   t
LEFT JOIN t n
       ON  n.PolicyChainRef = t.PolicyChainRef AND n.product = t.product AND n.chain_seq = t.chain_seq + 1
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 5, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Renewal episodes: one per term reaching its renewal (due_flag = 1).

In [5]:
CREATE OR REPLACE TABLE stg.fe3_episode_renewal AS
SELECT concat('RN|', o.ProductName, '|', o.policy_code, '|', CAST(o.due_date AS STRING))   AS EpisodeID,
       o.SCV_Customer_Key                                                               AS scv_id,
       o.policy_code                                                                    AS PolicyRef,
       o.ProductName                                                                    AS Product,
       'Renewal'                                                                        AS EpisodeType,
       date_sub(o.due_date, 60)                                                         AS EpisodeStart,
       date_add(o.due_date, 14)                                                         AS EpisodeEnd,
       CAST(NULL AS DATE)                                                               AS SaleDate,
       o.due_date                                                                       AS DueDate,
       o.DecisionDate, o.DecisionType, o.PolicyChainRef,
       o.term_outcome                                                                   AS Outcome_ThisRenewal,
       o.term_outcome_date                                                              AS Outcome_ThisRenewal_Date,
       CASE WHEN o.term_outcome IN ('Lapsed', 'Cancelled') THEN NULL
            WHEN o.term_outcome = 'Not yet known' OR o.next_term_outcome IS NULL THEN 'Not yet known'
            ELSE o.next_term_outcome END                                                AS Outcome_NextRenewal,
       CASE WHEN o.term_outcome IN ('Lapsed', 'Cancelled', 'Not yet known') THEN NULL
            ELSE o.next_term_outcome_date END                                           AS Outcome_NextRenewal_Date,
       o.new_policy_code, o.new_due_date, o.match_evidence, o.chain_seq, o.outcome_status
FROM   stg.fe3_term_outcome o
WHERE  o.due_flag = 1 AND o.SCV_Customer_Key IS NOT NULL
AND    date_sub(o.due_date, 60) >= ${FirstEpisodeDate}
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 6, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

First-year episodes: New Business sales of Car / Home / Van that are genuinely new: not Temporary / NB Not Taken Up, and not the replacement in a renewal or rebook link (a lapse or cancellation rebought within 30 days). The first term is the policy's term whose renewal date is nearest the sale's (within 31 days).

**Notes**

- Real New Business sales only (SaleCount \>= 1), one per policy: its earliest. EDW's 'Cat C: Cancelled Sale' holds both sales cancelled in their first month (SaleCount 1, kept) and cancellation records (SaleCount 0, e.g. JOHJ7A001's January 2024 row); corrections are 0 or -1.
- EpisodeStart (spec): the first quote linked to the sale, no earlier than 30 days before it. online (Motor MFQ): the bought quote (stg.GlobalPoliciesSold.QuoteQueryGuid) -\> its first event in the stream Home online / web assist: the HEQ quote EDW links to the policy (edw.tbl_fact_policy_quotes, ~39% of Home NB) back office (Motor system 9, Home system 4): the quote created with the Relay policy (scveadmlinkfinal.QuoteStartDate) otherwise (Van, the rest): the earliest acquisition event of the customer and product in the 30 days before the sale
- Home online / web-assist quotes linked to the policy by EDW (matched on email, mobile or address)
- The first renewal's outcome. A first year that never reached the stock (cancelled within its first month) has no term: Cancelled before renewal when ods.policy holds a cancellation, else Not yet known.

In [6]:
CREATE OR REPLACE TABLE stg.fe3_episode_firstyear AS
WITH nb_all AS (
    SELECT TRIM(s.PolicyCode) AS PolicyRef, s.ReportingSaleDate AS SaleDate, s.PolicyRenewDateAdj AS sale_due,
           p.product, p.PolicyCancelDate, c.SCV_Customer_Key AS scv_id,
           ROW_NUMBER() OVER (PARTITION BY TRIM(s.PolicyCode) ORDER BY s.ReportingSaleDate) AS rn
    FROM   edw.tbl_fact_policy_sales s
    JOIN   stg.pr30_policy p ON p.PolicyCode = TRIM(s.PolicyCode)
    LEFT JOIN stg.pr30_scv_client c ON c.ClientCode = p.ClientCode
    WHERE  s.ReportingSaleType = 'New Business'
    AND    COALESCE(s.SaleCount, 0) >= 1
    AND    p.product IN ('Motor', 'Home', 'Van')
    AND    COALESCE(p.PolicyStatusDesc, '') NOT IN ('Temporary', 'NB Not Taken Up')
),
nb AS (
    SELECT * FROM nb_all WHERE rn = 1 AND SaleDate >= date_sub(${FirstEpisodeDate}, 30)
),
replaced AS (
    SELECT DISTINCT new_policy_code, new_sale_date FROM stg.seg_policy_renewals_30d WHERE link_type IN ('renewal', 'rebook')
),
genuine AS (
    SELECT nb.* FROM nb
    LEFT JOIN replaced r ON r.new_policy_code = nb.PolicyRef AND r.new_sale_date = nb.SaleDate
    WHERE r.new_policy_code IS NULL AND nb.scv_id IS NOT NULL
),
first_term AS (
    SELECT g.PolicyRef, g.SaleDate, o.due_date, o.PolicyChainRef, o.term_outcome, o.term_outcome_date,
           ROW_NUMBER() OVER (PARTITION BY g.PolicyRef, g.SaleDate ORDER BY abs(datediff(o.due_date, g.sale_due))) AS rn
    FROM   genuine g
    JOIN   stg.fe3_term_outcome o ON o.policy_code = g.PolicyRef AND abs(datediff(o.due_date, g.sale_due)) <= 31
),
online_q AS (
    SELECT g.PolicyRef, g.SaleDate, MIN(e.EventDate) AS d
    FROM   genuine g
    JOIN   (SELECT DISTINCT TRIM(PolicyCode) AS PolicyCode, QuoteQueryGuid FROM stg.globalpoliciessold
            WHERE QuoteQueryGuid IS NOT NULL) q ON q.PolicyCode = g.PolicyRef
    JOIN   ods.episodeeventsnumberedfinal e
           ON e.SourceQuoteReference = q.QuoteQueryGuid AND e.EpisodeType = 'acquisition'
    GROUP  BY g.PolicyRef, g.SaleDate
),
heq_q AS (
    SELECT TRIM(PolicyCode) AS PolicyRef, MIN(CAST(HEQCreatedDatetime AS DATE)) AS d
    FROM   edw.tbl_fact_policy_quotes
    WHERE  PolicyTypeGroup = 'Home' AND HEQQuotationID IS NOT NULL AND HEQCreatedDatetime IS NOT NULL
    GROUP  BY TRIM(PolicyCode)
),
backoffice_q AS (
    SELECT TRIM(ChillSourcePolicyReference) AS PolicyRef, MIN(CAST(substr(QuoteStartDate, 1, 10) AS DATE)) AS d
    FROM   ods.scveadmlinkfinal
    WHERE  SourceSystemId IN (4, 9) AND ChillSourcePolicyReference IS NOT NULL AND QuoteStartDate IS NOT NULL
    GROUP  BY TRIM(ChillSourcePolicyReference)
),
any_acq AS (
    SELECT g.PolicyRef, g.SaleDate, MIN(e.EventDate) AS d
    FROM   genuine g
    JOIN   ods.episodeeventsnumberedfinal e
           ON  e.SCV_Customer_Key = g.scv_id AND e.EpisodeType = 'acquisition'
           AND e.ProductType = lower(g.product)
           AND e.EventDate BETWEEN date_sub(g.SaleDate, 30) AND g.SaleDate
    GROUP  BY g.PolicyRef, g.SaleDate
),
acq AS (
    SELECT g.PolicyRef, g.SaleDate,
           greatest(date_sub(g.SaleDate, 30),
                    least(g.SaleDate, COALESCE(o.d, h.d, b.d, x.d, g.SaleDate)))              AS first_event,
           CASE WHEN o.d IS NOT NULL THEN 'online bought quote' WHEN h.d IS NOT NULL THEN 'HEQ quote'
                WHEN b.d IS NOT NULL THEN 'back-office quote'
                WHEN x.d IS NOT NULL THEN 'earliest acquisition event' ELSE 'sale date' END AS start_source
    FROM   genuine g
    LEFT JOIN online_q     o ON o.PolicyRef = g.PolicyRef AND o.SaleDate = g.SaleDate
    LEFT JOIN heq_q        h ON h.PolicyRef = g.PolicyRef AND g.product = 'Home'
    LEFT JOIN backoffice_q b ON b.PolicyRef = g.PolicyRef
    LEFT JOIN any_acq      x ON x.PolicyRef = g.PolicyRef AND x.SaleDate = g.SaleDate
),
dd AS (SELECT DataDate FROM stg.fe3_datadate)
SELECT concat('FY|', CASE g.product WHEN 'Motor' THEN 'Car' ELSE g.product END, '|', g.PolicyRef, '|',
              CAST(g.SaleDate AS STRING))                                               AS EpisodeID,
       g.scv_id,
       g.PolicyRef,
       CASE g.product WHEN 'Motor' THEN 'Car' ELSE g.product END                        AS Product,
       'First year'                                                                     AS EpisodeType,
       COALESCE(a.first_event, g.SaleDate)                                              AS EpisodeStart,
       date_sub(COALESCE(f.due_date, g.sale_due), 60)                                   AS EpisodeEnd,
       g.SaleDate,
       COALESCE(f.due_date, g.sale_due)                                                 AS DueDate,
       CAST(NULL AS DATE)                                                               AS DecisionDate,
       CAST(NULL AS STRING)                                                             AS DecisionType,
       COALESCE(f.PolicyChainRef, concat(g.PolicyRef, '|', CAST(g.sale_due AS STRING))) AS PolicyChainRef,
       CAST(NULL AS STRING)                                                             AS Outcome_ThisRenewal,
       CAST(NULL AS DATE)                                                               AS Outcome_ThisRenewal_Date,
       CASE WHEN f.term_outcome IS NOT NULL THEN f.term_outcome
            WHEN CAST(g.PolicyCancelDate AS DATE) BETWEEN g.SaleDate AND g.sale_due THEN 'Cancelled before renewal'
            ELSE 'Not yet known' END                                                    AS Outcome_NextRenewal,
       CASE WHEN f.term_outcome IS NOT NULL THEN f.term_outcome_date
            WHEN CAST(g.PolicyCancelDate AS DATE) BETWEEN g.SaleDate AND g.sale_due THEN CAST(g.PolicyCancelDate AS DATE)
            END                                                                         AS Outcome_NextRenewal_Date,
       CAST(NULL AS STRING) AS new_policy_code, CAST(NULL AS DATE) AS new_due_date, CAST(NULL AS STRING) AS match_evidence,
       CAST(NULL AS INT) AS chain_seq, CAST(NULL AS STRING) AS outcome_status
FROM   genuine g
LEFT JOIN first_term f ON f.PolicyRef = g.PolicyRef AND f.SaleDate = g.SaleDate AND f.rn = 1
LEFT JOIN acq a        ON a.PolicyRef = g.PolicyRef AND a.SaleDate = g.SaleDate
WHERE  COALESCE(a.first_event, g.SaleDate) >= ${FirstEpisodeDate}
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 7, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Change and Document request episodes (Car and Home change; Van change Unread: no source)

Genesys queue map (Vijay, 2026-10-13: out/genesys_queue_map.csv). In-scope queues only; QD\_\* (Quote Devil), chats, emails, tests, SME, Life and calls with no queue are left out.

In [7]:
CREATE OR REPLACE TABLE stg.fe3_queue_map AS
SELECT * FROM VALUES
    ('INBOUND_Car_Renewals', 'Motor', 'renewal'),
    ('INBOUND_Documents_In', CAST(NULL AS STRING), 'document'),
    ('INBOUND_Documents_Out', CAST(NULL AS STRING), 'document'),
    ('INBOUND_Home_Add_Item', 'Home', 'change'),
    ('INBOUND_Home_Renewals', 'Home', 'renewal'),
    ('INBOUND_HomeChangeBuildCost', 'Home', 'change'),
    ('INBOUND_HomeChangeOther', 'Home', 'change'),
    ('INBOUND_HomeMortgage', 'Home', 'change'),
    ('INBOUND_IntensiveCare_Renewals', CAST(NULL AS STRING), 'renewal'),
    ('INBOUND_NON-STANDARD_HOME', 'Home', 'sales'),
    ('INBOUND_NON-STANDARD_MOTOR', 'Motor', 'sales'),
    ('INBOUND_SALES_HOME', 'Home', 'sales'),
    ('INBOUND_SALES_MOTOR', 'Motor', 'sales'),
    ('INBOUND_SALES_VAN', 'Van', 'sales'),
    ('INBOUND_Van_Renewals', 'Van', 'renewal'),
    ('INBOUND_Vehicle_Add_Driver', 'Motor', 'change'),
    ('INBOUND_VehicleChange_Perm', 'Motor', 'change'),
    ('INBOUND_VehicleChange_Temp', 'Motor', 'change'),
    ('INBOUND_VehicleChangeAddress', 'Motor', 'change'),
    ('INBOUND_VehicleChangeOther', 'Motor', 'change'),
    ('INBOUND_VehicleUpdateLicence', 'Motor', 'change'),
    ('OUTBOUND_SALES_HOME', 'Home', 'sales'),
    ('OUTBOUND_SALES_LIFE', 'Life', 'sales'),
    ('OUTBOUND_SALES_MOTOR', 'Motor', 'sales'),
    ('OUTBOUND_SALES_VAN', 'Van', 'sales')
AS t(queue_name, product, spec_line)
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Inbound calls, one row per conversation, from dlk.genesys_session_summary; customer from ods.customercallevents. Line: renewal if any session reached a renewal queue (transfers from sales / customer service count as renewal calls), else change, else document, else sales. queue_product = the deciding queue's product (NULL = by rule).

In [8]:
CREATE OR REPLACE TABLE stg.fe3_calls AS
WITH s AS (
    SELECT g.conversationId, g.conversationStartTime, m.spec_line, m.product,
           CASE m.spec_line WHEN 'renewal' THEN 1 WHEN 'change' THEN 2 WHEN 'document' THEN 3 WHEN 'sales' THEN 4 END AS rk
    FROM   dlk.genesys_session_summary g
    JOIN   stg.fe3_queue_map m ON m.queue_name = g.queueName
    WHERE  g.originatingDirection = 'inbound'
    AND    g.conversationStartTime >= date_sub(${FirstEpisodeDate}, 60)
),
c AS (
    SELECT conversationId, MIN(conversationStartTime) AS call_ts, MIN(rk) AS rk
    FROM   s GROUP BY conversationId
),
p AS (
    SELECT s.conversationId, MAX(s.product) AS queue_product
    FROM   s JOIN c ON c.conversationId = s.conversationId AND c.rk = s.rk
    GROUP  BY s.conversationId
),
k AS (
    SELECT conversationId, MAX(SCV_Customer_Key) AS scv_id
    FROM   ods.customercallevents WHERE originatingDirection = 'inbound' AND SCV_Customer_Key > 0
    GROUP  BY conversationId
)
SELECT c.conversationId, c.call_ts, CAST(c.call_ts AS DATE) AS call_date,
       CASE c.rk WHEN 1 THEN 'renewal' WHEN 2 THEN 'change' WHEN 3 THEN 'document' ELSE 'sales' END AS line,
       p.queue_product, k.scv_id
FROM   c
LEFT JOIN p ON p.conversationId = c.conversationId
LEFT JOIN k ON k.conversationId = c.conversationId
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 9, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Live policy terms per customer: a term is live from its start to its end (cancellation, else renewal date).

In [9]:
CREATE OR REPLACE TABLE stg.fe3_live_terms AS
SELECT SCV_Customer_Key AS scv_id, product, ProductName, policy_code, due_date, PolicyChainRef,
       add_months(due_date, -12) AS live_from, COALESCE(cancel_date, due_date) AS live_to
FROM   stg.fe3_terms
WHERE  SCV_Customer_Key IS NOT NULL
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 10, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Car change input: change-line calls attached to the customer's live Motor terms on the call date (all of them if several: spec DE3), plus the policy's change events M3 / M4a / M4c / M6 (M5 has no events, so it is ignored).

**Notes**

- One term per policy and date: a policy whose renewal date moved can have two overlapping terms; v1 attached the same call to both (288 duplicate episode ids). The earliest term still live wins.
- `WHERE c.line = 'change' AND c.queue_product IN ('Motor', 'Home')`: Home change calls (v2); Van: none (Unread)
- `WHERE due_date = first_due`: v3: one term per policy and date, whatever the event (v2 chose per event type)

In [10]:
CREATE OR REPLACE TABLE stg.fe3_change_input AS
WITH u AS (
SELECT t.ProductName, t.policy_code, t.due_date, t.PolicyChainRef, t.scv_id, c.call_date AS event_date, 'call' AS kind
FROM   stg.fe3_calls c
JOIN   stg.fe3_live_terms t
       ON  t.scv_id = c.scv_id AND t.product = c.queue_product
       AND c.call_date >= t.live_from AND c.call_date < t.live_to
WHERE  c.line = 'change' AND c.queue_product IN ('Motor', 'Home')
UNION ALL
SELECT t.ProductName, t.policy_code, t.due_date, t.PolicyChainRef, t.scv_id, e.EventDate, e.EventTypeId
FROM   (SELECT DISTINCT TRIM(SourcePolicyReference) AS pc, EventDate, EventTypeId
        FROM ods.episodeeventsnumberedfinal
        WHERE EpisodeType = 'mta' AND ProductType = 'motor' AND EventTypeId IN ('M3', 'M4a', 'M4c', 'M6')
        AND SourcePolicyReference IS NOT NULL AND EventDate >= date_sub(${FirstEpisodeDate}, 60)) e
JOIN   stg.fe3_live_terms t
       ON  t.policy_code = e.pc AND t.product = 'Motor'
       AND e.EventDate >= t.live_from AND e.EventDate <= t.live_to
)
SELECT DISTINCT ProductName, policy_code, due_date, PolicyChainRef, scv_id, event_date, kind
FROM  (SELECT u.*, MIN(due_date) OVER (PARTITION BY ProductName, policy_code, event_date) AS first_due
       FROM u) x
WHERE  due_date = first_due
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 11, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

In [11]:
%%pyspark
# Change episodes, per policy term. Home (v2) has calls only, so each episode ends 30 days after its last call.
# Car (agreed 2026-10-13): a change call starts an episode; further calls within 30
# days of the previous one join it; the first M3 / M4a / M4c / M6 on or after the start ends it; with no ending event
# within 30 days of the last call it ends 30 days after that call. An M3 with no open call episode starts its own
# episode, ended by the next M4a / M4c / M6 within 30 days, else on the M3 date.
import datetime as dt
import pandas as pd
from pyspark.sql import types as T

inp = spark.table("stg.fe3_change_input")
schema = T.StructType([T.StructField("ProductName", T.StringType()),
                       T.StructField("policy_code", T.StringType()), T.StructField("due_date", T.DateType()),
                       T.StructField("PolicyChainRef", T.StringType()), T.StructField("scv_id", T.LongType()),
                       T.StructField("EpisodeStart", T.DateType()), T.StructField("EpisodeEnd", T.DateType()),
                       T.StructField("end_event", T.StringType()), T.StructField("calls", T.IntegerType()),
                       T.StructField("started_by", T.StringType())])
D30 = dt.timedelta(days=30)
ENDS = ("M3", "M4a", "M4c", "M6")


def episodes(g):
    # calls before ending events on the same day, so a same-day call and acceptance make one episode
    g = g.assign(o=(g["kind"] != "call").astype(int)).sort_values(["event_date", "o"])
    out, ep = [], None

    def close(end, how):
        out.append((ep["start"], end, how, ep["calls"], ep["by"]))

    for d, k in zip(g["event_date"], g["kind"]):
        if ep is not None and d > ep["limit"]:
            close(ep["last"] + D30 if ep["by"] == "call" else ep["start"], "none")
            ep = None
        if k == "call":
            if ep is None:
                ep = {"start": d, "last": d, "limit": d + D30, "calls": 1, "by": "call"}
            elif ep["by"] == "call":
                ep["last"], ep["limit"], ep["calls"] = d, d + D30, ep["calls"] + 1
            else:                                   # a call during a no-call episode: close it, start a call episode
                close(ep["start"], "none")
                ep = {"start": d, "last": d, "limit": d + D30, "calls": 1, "by": "call"}
        elif k in ENDS:
            if ep is not None and (ep["by"] == "call" or k != "M3"):
                close(d, k)
                ep = None
            elif k == "M3":
                if ep is not None:
                    close(ep["start"], "none")
                ep = {"start": d, "last": d, "limit": d + D30, "calls": 0, "by": "M3"}
    if ep is not None:
        close(ep["last"] + D30 if ep["by"] == "call" else ep["start"], "none")
    return out


def per_partition(batches):
    parts = list(batches)
    if not parts:
        return
    pdf = pd.concat(parts, ignore_index=True)
    rows = []
    for (prod, pc, due, chain, scv), g in pdf.groupby(["ProductName", "policy_code", "due_date", "PolicyChainRef",
                                                          "scv_id"], sort=False):
        for st, en, how, n, by in episodes(g):
            rows.append((prod, pc, due, chain, int(scv), st, en, how, int(n), by))
    if rows:
        yield pd.DataFrame(rows, columns=[f.name for f in schema.fields])


(inp.repartition(200, "policy_code")
    .mapInPandas(per_partition, schema=schema)
    .write.mode("overwrite").format("delta").option("overwriteSchema", "true")
    .saveAsTable("stg.fe3_change_raw"))

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 13, Finished, Available, Finished, False)

In [12]:
%%pyspark
# Document request episodes, per customer (spec): document-line calls join while each is within 14 days of the
# previous one; the episode ends 14 days after the last call.
import datetime as dt
import pandas as pd
from pyspark.sql import types as T

calls = spark.sql("SELECT scv_id, call_date FROM stg.fe3_calls WHERE line = 'document' AND scv_id IS NOT NULL")
schema = T.StructType([T.StructField("scv_id", T.LongType()), T.StructField("EpisodeStart", T.DateType()),
                       T.StructField("EpisodeEnd", T.DateType()), T.StructField("calls", T.IntegerType())])
D14 = dt.timedelta(days=14)


def per_partition(batches):
    parts = list(batches)
    if not parts:
        return
    pdf = pd.concat(parts, ignore_index=True).sort_values(["scv_id", "call_date"])
    rows = []
    for scv, g in pdf.groupby("scv_id", sort=False):
        start = last = None
        n = 0
        for d in g["call_date"]:
            if start is not None and d - last <= D14:
                last, n = d, n + 1
            else:
                if start is not None:
                    rows.append((int(scv), start, last + D14, n))
                start, last, n = d, d, 1
        rows.append((int(scv), start, last + D14, n))
    if rows:
        yield pd.DataFrame(rows, columns=[f.name for f in schema.fields])


(calls.repartition(200, "scv_id")
      .mapInPandas(per_partition, schema=schema)
      .write.mode("overwrite").format("delta").option("overwriteSchema", "true")
      .saveAsTable("stg.fe3_docreq_raw"))

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 14, Finished, Available, Finished, False)

Change episodes (Car, Home). Outcome_NextRenewal: the first term of the same chain due after EpisodeEnd.

In [13]:
CREATE OR REPLACE TABLE stg.fe3_episode_change AS
WITH n AS (
    SELECT c.*, t.term_outcome, t.term_outcome_date,
           ROW_NUMBER() OVER (PARTITION BY c.ProductName, c.policy_code, c.due_date, c.EpisodeStart
                              ORDER BY t.due_date) AS rn
    FROM   stg.fe3_change_raw c
    LEFT JOIN stg.fe3_term_outcome t ON t.PolicyChainRef = c.PolicyChainRef AND t.due_date > c.EpisodeEnd
)
SELECT concat('CH|', ProductName, '|', policy_code, '|', CAST(EpisodeStart AS STRING)) AS EpisodeID,
       scv_id, policy_code AS PolicyRef, ProductName AS Product, 'Change' AS EpisodeType,
       EpisodeStart, EpisodeEnd, CAST(NULL AS DATE) AS SaleDate, CAST(NULL AS DATE) AS DueDate,
       CAST(NULL AS DATE) AS DecisionDate, CAST(NULL AS STRING) AS DecisionType, PolicyChainRef,
       CAST(NULL AS STRING) AS Outcome_ThisRenewal, CAST(NULL AS DATE) AS Outcome_ThisRenewal_Date,
       COALESCE(term_outcome, 'Not yet known')                               AS Outcome_NextRenewal,
       term_outcome_date                                                     AS Outcome_NextRenewal_Date,
       CAST(NULL AS STRING) AS new_policy_code, CAST(NULL AS DATE) AS new_due_date,
       concat('ended by ', end_event, '; started by ', started_by, '; calls ', CAST(calls AS STRING)) AS match_evidence,
       CAST(NULL AS INT) AS chain_seq, CAST(NULL AS STRING) AS outcome_status
FROM   n
WHERE  rn = 1 AND EpisodeStart >= ${FirstEpisodeDate}
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 15, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

Document request episodes. Product by the spec's rule: the customer's only live Car / Home / Van term on the first call date (wrap codes not yet analysed); otherwise Unattributed (kept and counted, left out of product measures).

In [14]:
CREATE OR REPLACE TABLE stg.fe3_episode_docreq AS
WITH live AS (
    SELECT d.scv_id, d.EpisodeStart, d.EpisodeEnd, d.calls, t.policy_code, t.ProductName, t.PolicyChainRef,
           COUNT(t.policy_code) OVER (PARTITION BY d.scv_id, d.EpisodeStart)                  AS n_live,
           ROW_NUMBER() OVER (PARTITION BY d.scv_id, d.EpisodeStart ORDER BY t.policy_code)  AS rn
    FROM   stg.fe3_docreq_raw d
    LEFT JOIN stg.fe3_live_terms t
           ON t.scv_id = d.scv_id AND d.EpisodeStart >= t.live_from AND d.EpisodeStart < t.live_to
),
one AS (
    SELECT scv_id, EpisodeStart, EpisodeEnd, calls,
           CASE WHEN n_live = 1 THEN policy_code END                     AS policy_code,
           CASE WHEN n_live = 1 THEN ProductName ELSE 'Unattributed' END AS Product,
           CASE WHEN n_live = 1 THEN PolicyChainRef END                  AS PolicyChainRef
    FROM   live WHERE rn = 1
),
n AS (
    SELECT o.*, t.term_outcome, t.term_outcome_date,
           ROW_NUMBER() OVER (PARTITION BY o.scv_id, o.EpisodeStart ORDER BY t.due_date) AS rn2
    FROM   one o
    LEFT JOIN stg.fe3_term_outcome t ON t.PolicyChainRef = o.PolicyChainRef AND t.due_date > o.EpisodeEnd
)
SELECT concat('DR|', CAST(scv_id AS STRING), '|', CAST(EpisodeStart AS STRING))   AS EpisodeID,
       scv_id, policy_code AS PolicyRef, Product, 'Document request' AS EpisodeType,
       EpisodeStart, EpisodeEnd, CAST(NULL AS DATE) AS SaleDate, CAST(NULL AS DATE) AS DueDate,
       CAST(NULL AS DATE) AS DecisionDate, CAST(NULL AS STRING) AS DecisionType, PolicyChainRef,
       CAST(NULL AS STRING) AS Outcome_ThisRenewal, CAST(NULL AS DATE) AS Outcome_ThisRenewal_Date,
       CASE WHEN policy_code IS NULL THEN NULL ELSE COALESCE(term_outcome, 'Not yet known') END AS Outcome_NextRenewal,
       term_outcome_date                                                                     AS Outcome_NextRenewal_Date,
       CAST(NULL AS STRING) AS new_policy_code, CAST(NULL AS DATE) AS new_due_date,
       concat('calls ', CAST(calls AS STRING)) AS match_evidence,
       CAST(NULL AS INT) AS chain_seq, CAST(NULL AS STRING) AS outcome_status
FROM   n
WHERE  rn2 = 1 AND EpisodeStart >= ${FirstEpisodeDate}
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 16, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

The frame: First year, Renewal, Change (Car, Home) and Document request episodes, with DataDate.

In [15]:
CREATE OR REPLACE TABLE stg.fe3_episode AS
SELECT e.*, (SELECT DataDate FROM stg.fe3_datadate) AS DataDate
FROM   (SELECT * FROM stg.fe3_episode_renewal UNION ALL SELECT * FROM stg.fe3_episode_firstyear
        UNION ALL SELECT * FROM stg.fe3_episode_change UNION ALL SELECT * FROM stg.fe3_episode_docreq) e
;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 17, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Checks

Grain (expect 0) and DataDate.

In [16]:
SELECT COUNT(*) AS episodes, COUNT(*) - COUNT(DISTINCT EpisodeID) AS duplicate_ids, MAX(DataDate) AS DataDate,
       SUM(CASE WHEN EpisodeStart > EpisodeEnd THEN 1 ELSE 0 END) AS start_after_end
FROM   stg.fe3_episode;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 18, Finished, Available, Finished, False)

<Spark SQL result set with 1 rows and 4 fields>

Episodes by snapshot (by EpisodeStart), product and type, with outcomes.

In [17]:
SELECT CASE WHEN EpisodeStart BETWEEN '2023-07-01' AND '2024-06-30' THEN '202406'
            WHEN EpisodeStart BETWEEN '2024-07-01' AND '2025-06-30' THEN '202506'
            WHEN EpisodeStart BETWEEN '2025-07-01' AND '2026-06-30' THEN '202606'
            ELSE 'outside' END AS snapshot,
       Product, EpisodeType, COUNT(*) AS episodes,
       SUM(CASE WHEN Outcome_ThisRenewal = 'Renewed' THEN 1 ELSE 0 END)       AS this_renewed,
       SUM(CASE WHEN Outcome_ThisRenewal IN ('Lapsed', 'Cancelled') THEN 1 ELSE 0 END) AS this_lost,
       SUM(CASE WHEN Outcome_ThisRenewal = 'Not yet known' THEN 1 ELSE 0 END) AS this_not_yet_known,
       SUM(CASE WHEN Outcome_NextRenewal = 'Renewed' THEN 1 ELSE 0 END)       AS next_renewed,
       SUM(CASE WHEN Outcome_NextRenewal IN ('Lapsed', 'Cancelled', 'Cancelled before renewal') THEN 1 ELSE 0 END) AS next_lost,
       SUM(CASE WHEN Outcome_NextRenewal = 'Rebooked' THEN 1 ELSE 0 END)      AS next_rebooked,
       SUM(CASE WHEN Outcome_NextRenewal = 'Not yet known' THEN 1 ELSE 0 END) AS next_not_yet_known
FROM   stg.fe3_episode
GROUP  BY 1, 2, 3 ORDER BY 1, 2, 3;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 19, Finished, Available, Finished, False)

<Spark SQL result set with 48 rows and 11 fields>

Change and document request episodes: lengths, how they ended, attribution.

In [18]:
SELECT EpisodeType, Product, COUNT(*) AS episodes,
       ROUND(AVG(datediff(EpisodeEnd, EpisodeStart)), 1) AS avg_days,
       SUM(CASE WHEN match_evidence LIKE 'ended by M%' THEN 1 ELSE 0 END) AS ended_by_event,
       SUM(CASE WHEN Outcome_NextRenewal = 'Renewed' THEN 1 ELSE 0 END)  AS next_renewed,
       SUM(CASE WHEN Outcome_NextRenewal IS NULL THEN 1 ELSE 0 END)      AS next_blank
FROM   stg.fe3_episode WHERE EpisodeType IN ('Change', 'Document request')
GROUP  BY 1, 2 ORDER BY 1, 2;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 20, Finished, Available, Finished, False)

<Spark SQL result set with 6 rows and 7 fields>

Decision types on renewal episodes.

In [19]:
SELECT Product, DecisionType, Outcome_ThisRenewal, COUNT(*) AS episodes
FROM   stg.fe3_episode WHERE EpisodeType = 'Renewal'
GROUP  BY 1, 2, 3 ORDER BY 1, 2, 3;

StatementMeta(, c6bcc168-faf6-42ce-b3c2-dbba9911a74e, 21, Finished, Available, Finished, False)

<Spark SQL result set with 12 rows and 4 fields>